# 04 — Business Analysis | Customer Intelligence
**Projeto fictício:** iFood Customer Analytics · **Demanda:** MKT-001 · **Solicitante:** Marketing & CRM

## 1. Contexto e objetivo
Priorizar públicos e canais de comunicação com orçamento limitado, usando comportamento observado e resposta histórica. A entrega não é uma previsão causal nem uma lista pronta para disparo. A unidade de análise é o registro deduplicado, sem chave de cliente confirmada. A fonte operacional é exclusivamente o Parquet do notebook 02.

**Perguntas:** onde se concentra o consumo? Quais grupos combinam valor e atividade? Como variam as respostas históricas? Quais ações merecem experimentação?

**Referências internas:** Desafio Ifood - Instruções, p. 1–2; notebooks 01–03; relatório de limpeza. Renda familiar anual; gastos por categoria nos últimos dois anos; recência em dias; visitas web no último mês. Moeda e janela das contagens de compras não informadas.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
ROOT = Path('..') if Path('../data/processed').exists() else Path('.')
SOURCE = ROOT / 'data/processed/mkt_data_clean.parquet'
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures' / 'business'
FIGURES.mkdir(parents=True, exist_ok=True)
RED, DARK, GRAY = '#EA1D2C', '#303238', '#A5AAB2'
sns.set_theme(style='whitegrid', palette=[RED, DARK, GRAY])
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
def finish(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / (name + '.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 2. Contrato de entrada e validação
O notebook não refaz a limpeza. Falhas no contrato interrompem a análise; alterações de versão devem ser reconciliadas com o notebook 02.

**Nota de validação desta entrega:** as saídas salvas foram produzidas sobre o CSV original reconstruído com as regras do notebook 02, não sobre o Parquet local. Execute todas as células com seu Parquet para validar a versão definitiva.

In [ ]:
df = pd.read_parquet(SOURCE)
print('Base recebida:', df.shape)


In [3]:
campaigns = [f'AcceptedCmp{i}' for i in range(1,6)] + ['Response']
channels = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases']
products = ['MntWines','MntMeatProducts','MntFishProducts','MntFruits','MntSweetProducts','MntGoldProds']
required = set(campaigns + channels + products + ['Income','expenses','MntTotal','MntRegularProds','Recency','kids','NumDealsPurchases','NumWebVisitsMonth','education_level','marital_status','AcceptedCmpOverall'])
assert required <= set(df.columns), required-set(df.columns)
assert df.shape == (2021,31), 'Versão diferente da base aprovada.'
assert not df.isna().any().any() and not df.duplicated().any()
assert df[campaigns].isin([0,1]).all().all()
assert (df[channels+products+['Recency','expenses']] >= 0).all().all()
assert df.expenses.eq(df.MntTotal).all()
work = df.copy()
print('Contrato aprovado.')


Contrato aprovado.


## 3. Reconciliação de consumo
Antes de chamar uma medida de gasto total, verificamos sua composição. Não somamos `expenses` a `MntTotal`, nem presumimos que Gold esteja incluído.

In [4]:
checks = pd.Series({'expenses = cinco categorias': df.expenses.eq(df[products[:-1]].sum(axis=1)).mean(), 'expenses = seis categorias': df.expenses.eq(df[products].sum(axis=1)).mean(), 'AcceptedCmpOverall = cinco campanhas': df.AcceptedCmpOverall.eq(df[campaigns[:-1]].sum(axis=1)).mean()})
display(checks.to_frame('proporção de igualdade'))
assert checks.iloc[0] == 1
work['spend_all'] = work[products].sum(axis=1)
work['frequency_proxy'] = work[channels].sum(axis=1)
work['prior_acceptance'] = work[campaigns[:-1]].sum(axis=1)


,proporção de igualdade
expenses = cinco categorias,1.000000
expenses = seis categorias,0.028204
AcceptedCmpOverall = cinco campanhas,1.000000


**Decisão:** `expenses` será o indicador principal para compatibilidade com a EDA; soma das seis categorias será uma medida complementar. `frequency_proxy` soma compras registradas em três canais, sem adicionar compras com desconto, pois elas podem sobrepor-se aos canais. Não é uma taxa temporal nem uma frequência validada de pedidos únicos.

## 4. KPIs executivos
$$\bar M=\frac{1}{n}\sum M_i,\quad p=\frac{\sum y_i}{n}$$
A resposta registrada é a proporção de registros com flag positiva, não uma taxa entre destinatários confirmados. Não há receita da empresa, margem ou custo causal para ROI.

In [5]:
kpis = pd.Series({'registros':len(work),'gasto_cinco_categorias':work.expenses.sum(),'gasto_seis_categorias':work.spend_all.sum(),'gasto_medio':work.expenses.mean(),'gasto_mediano':work.expenses.median(),'renda_familiar_anual_mediana':work.Income.median(),'recencia_mediana_dias':work.Recency.median(),'compras_proxy_media':work.frequency_proxy.mean(),'resposta_ultima_pct':100*work.Response.mean()}).rename_axis('metric').reset_index(name='value')
display(kpis.round(2))


,metric,value
0,registros,2021.00
1,gasto_cinco_categorias,1139418.00
2,gasto_seis_categorias,1228184.00
3,gasto_medio,563.79
4,gasto_mediano,343.00
5,renda_familiar_anual_mediana,51412.00
6,recencia_mediana_dias,49.00
7,compras_proxy_media,12.57
8,resposta_ultima_pct,15.39


## 5. Pareto — concentração de consumo
Ordenamos gastos do maior ao menor. A participação acumulada é $C_k=\sum_{i=1}^k M_{(i)}/\sum_{i=1}^n M_i$. O corte de 20% usa arredondamento para cima; não pressupomos a regra 80/20.

In [6]:
ordered = work.expenses.sort_values(ascending=False).reset_index(drop=True)
pareto = pd.DataFrame({'records':np.arange(1,len(ordered)+1),'spend':ordered})
pareto['records_pct'] = 100*pareto.records/len(work)
pareto['spend_pct'] = 100*pareto.spend.cumsum()/ordered.sum()
cut = int(np.ceil(.2*len(work)))
top_share = pareto.loc[cut-1,'spend_pct']
needed = pareto.loc[pareto.spend_pct.ge(80),'records_pct'].iloc[0]
fig,ax=plt.subplots(figsize=(9,5)); ax.plot(np.r_[0,pareto.records_pct],np.r_[0,pareto.spend_pct],color=RED,lw=2); ax.plot([0,100],[0,100],'--',color=GRAY); ax.scatter([pareto.loc[cut-1,'records_pct']],[top_share],color=DARK); ax.axhline(80,ls=':',color=GRAY); ax.set(xlabel='Registros acumulados (%)',ylabel='Gastos acumulados (%)',title='Concentração de gastos — cinco categorias'); finish(fig,'pareto')
display(Markdown(f'**Leitura:** os {cut} maiores registros concentram {top_share:.1f}% dos gastos; atingir 80% exige {needed:.1f}% da base. Priorizar valor não significa abandonar os demais públicos.'))


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** os 405 maiores registros concentram 53.7% dos gastos; atingir 80% exige 36.8% da base. Priorizar valor não significa abandonar os demais públicos.

## 6. Segmentação RFM exploratória
Recência (menor é melhor), frequência aproximada e valor monetário (maior é melhor). Usamos medianas como cortes transparentes, inclusivos nos grupos altos/recentes. Os empates permanecem juntos. Não interpretamos os segmentos como churn, fidelidade comprovada ou propensão futura. Os limites dependem da amostra.

In [7]:
r_cut = work.Recency.median(); f_cut = work.frequency_proxy.median(); m_cut = work.expenses.median()
work['recent'] = work.Recency <= r_cut
work['frequent'] = work.frequency_proxy >= f_cut
work['high_value'] = work.expenses >= m_cut
work['segment'] = np.select([work.recent & work.frequent & work.high_value, ~work.recent & work.high_value, work.recent & work.high_value, work.recent & work.frequent, ~work.recent], ['Ativos de alto valor','Alto valor menos recente','Alto valor menor frequência','Ativos menor valor','Menor valor menos recente'], default='Recentes menor frequência')
print('Cortes:',r_cut,f_cut,m_cut)
segments = work.groupby('segment').agg(records=('expenses','size'),spend=('expenses','sum'),median_spend=('expenses','median'),median_recency=('Recency','median'),median_frequency=('frequency_proxy','median'),response_rate=('Response','mean')).reset_index()
segments['spend_share_pct']=100*segments.spend/segments.spend.sum(); segments['response_pct']=100*segments.response_rate
segments=segments.sort_values('spend',ascending=False)
assert segments.records.sum()==len(work)
display(segments.round(2))


Cortes: 49.0 12.0 343.0


,segment,records,spend,median_spend,median_recency,median_frequency,response_rate,spend_share_pct,response_pct
1,Alto valor menos recente,510,526971,960.5,73.0,18.0,0.15,46.25,14.51
2,Ativos de alto valor,496,508087,964.0,24.5,18.5,0.28,44.59,27.62
4,Menor valor menos recente,491,47617,60.0,75.0,6.0,0.04,4.18,4.28
5,Recentes menor frequência,478,37238,46.5,24.0,5.0,0.15,3.27,14.85
3,Ativos menor valor,39,11807,304.0,20.0,12.0,0.18,1.04,17.95
0,Alto valor menor frequência,7,7698,803.0,22.0,11.0,0.14,0.68,14.29


In [8]:
fig,ax=plt.subplots(figsize=(10,5)); segments.set_index('segment')[['spend_share_pct']].plot.barh(ax=ax,color=RED,legend=False); ax.invert_yaxis(); ax.set(xlabel='Participação nos gastos (%)',ylabel='',title='Valor concentrado por segmento'); finish(fig,'segment_value')
fig,ax=plt.subplots(figsize=(10,5)); segments.set_index('segment').response_pct.plot.barh(ax=ax,color=DARK); ax.axvline(100*work.Response.mean(),color=RED,ls='--',label='Base'); ax.legend(); ax.set(xlabel='Resposta registrada (%)',ylabel='',title='Resposta histórica por segmento'); finish(fig,'segment_response')
lead=segments.iloc[0]; display(Markdown(f'**Leitura:** {lead.segment} reúne {lead.records} registros e {lead.spend_share_pct:.1f}% do gasto. A resposta histórica não demonstra efeito incremental.'))


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** Alto valor menos recente reúne 510 registros e 46.2% do gasto. A resposta histórica não demonstra efeito incremental.

## 7. Campanhas e incerteza
Para cada flag calculamos proporção positiva e intervalo de Wilson de 95%, como referência sob hipótese de observações independentes. Não corrige viés de seleção nem mede elegibilidade. $$IC=\frac{\hat p+z^2/(2n)\pm z\sqrt{\hat p(1-\hat p)/n+z^2/(4n^2)}}{1+z^2/n}$$

In [9]:
def wilson(k,n,z=1.96):
    p=k/n; d=1+z*z/n; center=(p+z*z/(2*n))/d; half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return center-half,center+half
rows=[]
for col in campaigns:
    k=int(work[col].sum()); lo,hi=wilson(k,len(work)); rows.append((col,len(work),k,100*k/len(work),100*lo,100*hi))
performance=pd.DataFrame(rows,columns=['campaign','records','positive','positive_pct','ci_low_pct','ci_high_pct'])
display(performance.round(2))
fig,ax=plt.subplots(figsize=(9,5)); ax.bar(performance.campaign,performance.positive_pct,color=[GRAY]*5+[RED],yerr=[performance.positive_pct-performance.ci_low_pct,performance.ci_high_pct-performance.positive_pct],capsize=4); ax.set(ylabel='Flags positivas (%)',title='Aceitação registrada por campanha'); finish(fig,'campaigns')


,campaign,records,positive,positive_pct,ci_low_pct,ci_high_pct
0,AcceptedCmp1,2021,133,6.58,5.58,7.75
1,AcceptedCmp2,2021,26,1.29,0.88,1.88
2,AcceptedCmp3,2021,151,7.47,6.40,8.70
3,AcceptedCmp4,2021,155,7.67,6.59,8.91
4,AcceptedCmp5,2021,146,7.22,6.17,8.44
5,Response,2021,311,15.39,13.88,17.03


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
history = work.groupby(work.prior_acceptance.gt(0).rename('accepted_prior')).Response.agg(['size','sum','mean']).reset_index(); history['response_pct']=100*history['mean']; display(history.round(3))
fig,ax=plt.subplots(figsize=(7,4)); ax.bar(['Sem aceitação anterior','Com aceitação anterior'],history.response_pct,color=[GRAY,RED]); ax.set(ylabel='Resposta última (%)',title='Histórico de aceitação e resposta'); finish(fig,'campaign_history')
display(Markdown(f'**Leitura:** última campanha: {100*work.Response.mean():.2f}% de flags positivas. O histórico pode ajudar a formular públicos de teste, mas a cronologia e a disponibilidade das flags antes da decisão precisam ser confirmadas.'))


,accepted_prior,size,sum,mean,response_pct
0,False,1595,137,0.086,8.589
1,True,426,174,0.408,40.845


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** última campanha: 15.39% de flags positivas. O histórico pode ajudar a formular públicos de teste, mas a cronologia e a disponibilidade das flags antes da decisão precisam ser confirmadas.

## 8. Canais de compra
Participação relativa das contagens registradas; não é conversão, pois não temos visitas por canal e janela compatível. Compras por catálogo e lojas não devem ser confundidas com pedidos atuais de delivery.

In [11]:
channel_summary=work[channels].sum().rename_axis('channel').reset_index(name='count'); channel_summary['share_pct']=100*channel_summary['count']/channel_summary['count'].sum(); display(channel_summary.round(2))
fig,ax=plt.subplots(figsize=(8,4)); ax.barh(['Web','Catálogo','Loja'],channel_summary.share_pct,color=RED); ax.set(xlabel='Participação das contagens (%)',title='Mix de canais'); finish(fig,'channels')
channel_segment=work.groupby('segment')[channels].sum(); channel_segment=channel_segment.div(channel_segment.sum(axis=1).replace(0,np.nan),axis=0)*100
fig,ax=plt.subplots(figsize=(10,5)); channel_segment.plot.barh(stacked=True,ax=ax,color=[RED,DARK,GRAY]); ax.set(xlabel='Participação (%)',ylabel='',title='Mix de canais por segmento'); finish(fig,'channels_segment')


,channel,count,share_pct
0,NumWebPurchases,8317,32.75
1,NumCatalogPurchases,5345,21.04
2,NumStorePurchases,11737,46.21


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Categorias e oportunidades de cross-selling
Usamos presença de gasto positivo nos dois anos como proxy de consumo de categoria. Não há cesta transacional, portanto não se trata de produtos comprados juntos. Lift acima de 1 indica coexistência superior à esperada sob independência, não causalidade.

In [12]:
mix=work[products].sum().rename_axis('category').reset_index(name='spend'); mix['share_pct']=100*mix.spend/mix.spend.sum(); mix=mix.sort_values('spend',ascending=False); display(mix.round(2))
fig,ax=plt.subplots(figsize=(9,5)); ax.barh(mix.category,mix.share_pct,color=RED); ax.invert_yaxis(); ax.set(xlabel='Participação no gasto de seis categorias (%)',title='Mix de produtos'); finish(fig,'product_mix')


,category,spend,share_pct
0,MntWines,619421,50.43
1,MntMeatProducts,335607,27.33
5,MntGoldProds,88766,7.23
2,MntFishProducts,75997,6.19
4,MntSweetProducts,55110,4.49
3,MntFruits,53283,4.34


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


$$\operatorname{lift}(A,B)=P(A\cap B)/(P(A)P(B))$$

In [13]:
presence=work[products].gt(0); pairs=[]
for a in products:
    for b in products:
        if a==b: continue
        pa=presence[a].mean(); pb=presence[b].mean(); joint=(presence[a]&presence[b]).mean(); eligible=int((presence[a]&~presence[b]).sum())
        pairs.append((a,b,joint,joint/pa if pa else np.nan,joint/(pa*pb) if pa*pb else np.nan,eligible))
cross=pd.DataFrame(pairs,columns=['source_category','target_category','support','confidence','lift','potential_records']).sort_values('lift',ascending=False)
display(cross.head(10).round(3))
fig,ax=plt.subplots(figsize=(9,6)); sns.heatmap(cross.pivot(index='source_category',columns='target_category',values='lift'),annot=True,fmt='.2f',cmap='Reds',ax=ax); ax.set(title='Coexistência de consumo — lift',xlabel='Categoria B',ylabel='Categoria A'); finish(fig,'cross_sell_lift')


,source_category,target_category,support,confidence,lift,potential_records
13,MntFishProducts,MntSweetProducts,0.731,0.886,1.090,190
22,MntSweetProducts,MntFishProducts,0.731,0.899,1.090,166
18,MntFruits,MntSweetProducts,0.725,0.883,1.086,195
23,MntSweetProducts,MntFruits,0.725,0.892,1.086,178
17,MntFruits,MntFishProducts,0.735,0.895,1.085,175
12,MntFishProducts,MntFruits,0.735,0.891,1.085,182
24,MntSweetProducts,MntGoldProds,0.798,0.981,1.010,31
29,MntGoldProds,MntSweetProducts,0.798,0.821,1.010,352
28,MntGoldProds,MntFruits,0.805,0.828,1.009,337
19,MntFruits,MntGoldProds,0.805,0.980,1.009,33


/tmp/ipykernel_542/829856281.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Robustez das decisões
Comparamos a concentração quando incluímos Gold e a classificação de alto valor sob corte alternativo no percentil 60. A coluna negativa `MntRegularProds` não é utilizada nas métricas primárias; não é corrigida neste notebook.

In [14]:
sensitivity=pd.DataFrame([{'measure':col,'top20_share_pct':100*work[col].nlargest(cut).sum()/work[col].sum()} for col in ['expenses','spend_all']]); display(sensitivity.round(2))
alt=work.expenses.ge(work.expenses.quantile(.6)); print('Mudança no indicador alto valor:',int((alt!=work.high_value).sum()),'registros')


,measure,top20_share_pct
0,expenses,53.67
1,spend_all,52.30


Mudança no indicador alto valor: 204 registros


## 11. Plano de ação e experimentação
Prioridade é uma decisão proposta, não um ranking de retorno esperado. O objetivo é testar benefício incremental com grupo controle, consentimento, limites de contato e identificação estável fora desta base. Evitar segmentar ofertas por atributos demográficos sensíveis ou inferir capacidade de pagamento individual pela renda familiar.

In [15]:
recommendations=pd.DataFrame([
('P1','Ativos de alto valor','Testar benefício de fidelização','Compra incremental por elegível','Margem, opt-out e custo'),
('P2','Alto valor menos recente','Testar comunicação de retorno','Reativação incremental','Margem e descadastro'),
('P3','Histórico de aceitação','Testar personalização versus mensagem padrão','Aceitação incremental','Reclamações e pressão de contato'),
('P4','Consumidores de categoria A sem B','Testar sugestão de categoria complementar','Compra incremental de B','Margem e canibalização')],columns=['priority','audience','action','primary_kpi','guardrails'])
display(recommendations)


,priority,audience,action,primary_kpi,guardrails
0,P1,Ativos de alto valor,Testar benefício de fidelização,Compra incremental por elegível,"Margem, opt-out e custo"
1,P2,Alto valor menos recente,Testar comunicação de retorno,Reativação incremental,Margem e descadastro
2,P3,Histórico de aceitação,Testar personalização versus mensagem padrão,Aceitação incremental,Reclamações e pressão de contato
3,P4,Consumidores de categoria A sem B,Testar sugestão de categoria complementar,Compra incremental de B,Margem e canibalização


### Desenho de teste A/B
Randomizar clientes elegíveis com identificador estável, estratificar por segmento, definir janela e métrica antes do início. Para planejamento ilustrativo, assumir proporções de 15% e 18%, alfa de 5% bilateral e poder de 80%. A aproximação normal de duas proporções é:
$$n=\frac{(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)}+z_{1-\beta}\sqrt{p_0(1-p_0)+p_1(1-p_1)})^2}{(p_1-p_0)^2}$$
Não confundir cenário de planejamento com impacto previsto.

In [16]:
from scipy.stats import norm
p0,p1=.15,.18; avg=(p0+p1)/2
n_arm=int(np.ceil((norm.ppf(.975)*np.sqrt(2*avg*(1-avg))+norm.ppf(.8)*np.sqrt(p0*(1-p0)+p1*(1-p1)))**2/(p1-p0)**2))
print(f'Amostra ilustrativa: {n_arm:,} por braço; {2*n_arm:,} no total. Base atual: {len(work):,}.')


Amostra ilustrativa: 2,402 por braço; 4,804 no total. Base atual: 2,021.


## 12. Publicação e rastreabilidade
Exportamos agregados, não identificadores inventados nem listas de contato. Os índices e parâmetros de classificação devem ser reavaliados em cada atualização.

In [17]:
exports={'business_kpis':kpis,'customer_segmentation':segments,'campaign_performance':performance,'campaign_history':history,'pareto_analysis':pareto,'channel_performance':channel_summary,'channel_by_segment':channel_segment.reset_index(),'business_product_mix':mix,'cross_sell_analysis':cross,'business_sensitivity':sensitivity,'business_recommendations':recommendations}
for name,table in exports.items(): table.to_csv(REPORTS/(name+'.csv'),index=False)
print(f'{len(exports)} relatórios exportados.')


11 relatórios exportados.


## 13. Parecer técnico e critérios de aceite
O resumo abaixo é calculado a partir da execução; recomendações são hipóteses, não resultados de campanhas.

In [18]:
display(Markdown(f"""### Resumo executivo
- **Concentração:** os 20% superiores (arredondados) representam {top_share:.1f}% dos gastos nas cinco categorias.
- **Segmento de maior valor agregado:** {lead.segment}, com {lead.records} registros e {lead.spend_share_pct:.1f}% do gasto.
- **Campanhas:** resposta registrada na última campanha de {100*work.Response.mean():.2f}%. Campanha anterior de maior proporção: {performance.iloc[:5].sort_values('positive_pct').iloc[-1].campaign}.
- **Canal dominante em contagens:** {channel_summary.sort_values('count').iloc[-1].channel}.
- **Categoria dominante no gasto das seis categorias:** {mix.iloc[0].category}.
- **Viabilidade experimental:** a amostra ilustrativa requerida é de {2*n_arm} registros independentes, acima da base atual.

### Limitações e riscos
Não há chave estável, exposição por campanha, margem, moeda, janela das compras por canal ou dados transacionais. O gasto de cinco categorias exclui Gold por construção observada. A deduplicação e as anomalias remanescentes são premissas herdadas do notebook 02. RFM é aproximação descritiva, sem validação temporal. Não é possível inferir ROI, churn, conversão ou causalidade.

### Critérios de aceite
- [x] Contrato de entrada avaliado nesta execução.
- [x] Composição de gastos reconciliada.
- [x] Segmentos mutuamente exclusivos e cobertura verificada.
- [x] Indicadores, gráficos e relatórios produzidos nesta execução.
- [ ] Validação das definições e elegibilidade com proprietário da fonte.
- [ ] Revisão de negócio e aprovação de experimentos.

**Próxima etapa:** apresentação executiva, com decisão de prioridade e plano de experimentação; nenhuma ativação automática."""))


### Resumo executivo
- **Concentração:** os 20% superiores (arredondados) representam 53.7% dos gastos nas cinco categorias.
- **Segmento de maior valor agregado:** Alto valor menos recente, com 510 registros e 46.2% do gasto.
- **Campanhas:** resposta registrada na última campanha de 15.39%. Campanha anterior de maior proporção: AcceptedCmp4.
- **Canal dominante em contagens:** NumStorePurchases.
- **Categoria dominante no gasto das seis categorias:** MntWines.
- **Viabilidade experimental:** a amostra ilustrativa requerida é de 4804 registros independentes, acima da base atual.

### Limitações e riscos
Não há chave estável, exposição por campanha, margem, moeda, janela das compras por canal ou dados transacionais. O gasto de cinco categorias exclui Gold por construção observada. A deduplicação e as anomalias remanescentes são premissas herdadas do notebook 02. RFM é aproximação descritiva, sem validação temporal. Não é possível inferir ROI, churn, conversão ou causalidade.

### Critérios de aceite
- [x] Contrato de entrada avaliado nesta execução.
- [x] Composição de gastos reconciliada.
- [x] Segmentos mutuamente exclusivos e cobertura verificada.
- [x] Indicadores, gráficos e relatórios produzidos nesta execução.
- [ ] Validação das definições e elegibilidade com proprietário da fonte.
- [ ] Revisão de negócio e aprovação de experimentos.

**Próxima etapa:** apresentação executiva, com decisão de prioridade e plano de experimentação; nenhuma ativação automática.